# Normalize single-token entropy by `ln|V|`

`data/out/single_token_entropy/` holds raw entropy in nats over each model's full
vocabulary. Those numbers are not comparable across models — the seven models here span
128256 to 200064 vocabulary entries — yet `EntropyGainSampler` subtracts a student's
entropy from a teacher proxy's. Dividing by `ln|V|` (the entropy of the uniform
distribution over the same support) puts every model on a common `[0, 1]` scale.

This notebook rewrites every parquet into `data/out/single_token_entropy_normalized/`
with **identical filenames and column names**, so consumers change only a path.

`|V|` comes from `AutoConfig(...).vocab_size` of the real model — the `lm_head` output
width, i.e. exactly the `logits.shape[-1]` that `compute_entropy_from_logits` divides by.
It is **not** `len(tokenizer)`: these models pad the embedding table past the tokenizer
(Qwen 151665 vs 151936, Phi-4-mini 200029 vs 200064). Both are printed below.

> **Note:** `core.complexity_estimation.entropy.logit_entropy.compute_entropy_from_logits`
> now normalizes at measurement time. So re-running anything under
> `src/experiments/estimate_single_token_entropy/` writes *already-normalized* values into
> the raw `data/out/single_token_entropy/` directory, and this notebook must not be run
> over that output — it would divide by `ln|V|` twice.

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoConfig, AutoTokenizer

from core.complexity_estimation.entropy.multi_token_entropy_estimator import (
    MultiTokenEntropyEstimatorSchema,
)
from core.complexity_estimation.entropy.single_token_entropy_estimator import (
    SingleTokenEntropyEstimatorSchema,
)

SOURCE_PATH = Path("../../data/out/single_token_entropy")
OUT_PATH = Path("../../data/out/single_token_entropy_normalized")

# Mirrors MODEL_NAME in src/experiments/estimate_single_token_entropy/{mmlu,gsm8k,gpqa}/<key>.py.
# Files are named "{dataset}_{key}.parquet", so the key is the suffix after the first underscore.
MODELS = {
    "llama_3b": "meta-llama/Llama-3.2-3B-Instruct",
    "llama_70b": "meta-llama/Llama-3.3-70B-Instruct",
    "mistral_24b": "mistralai/Mistral-Small-24B-Instruct-2501",
    "phi4mini": "microsoft/Phi-4-mini-instruct",
    "qwen_32b": "Qwen/Qwen2.5-32B-Instruct",
    "qwen_3b": "Qwen/Qwen2.5-3B-Instruct",
    "qwen_72b": "Qwen/Qwen2.5-72B-Instruct",
}

# Every entropy-valued field the two estimators write. scored_tokens is token ids, not entropy.
ENTROPY_COLUMNS = (
    set(SingleTokenEntropyEstimatorSchema.model_fields)
    | set(MultiTokenEntropyEstimatorSchema.model_fields)
) - {"scored_tokens"}

print("entropy columns:", sorted(ENTROPY_COLUMNS))

## Resolve `|V|` from the real models

Only `config.json` / tokenizer metadata is fetched — no weights. The gated `meta-llama`
repos need a Hugging Face token (`huggingface-cli login`).

In [ ]:
vocab_info = {}

for key, model_id in MODELS.items():
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)

    vocab_info[key] = {
        "model_id": model_id,
        "vocab_size": int(config.vocab_size),
        "tokenizer_len": len(tokenizer),
        "tokenizer_vocab_size": tokenizer.vocab_size,
        "ln_vocab_size": math.log(config.vocab_size),
    }

pd.DataFrame(vocab_info).T

## Normalize and write

In [ ]:
def normalize_column(column: pd.Series, divisor: float) -> pd.Series:
    """Divide a scalar entropy column, or a column of per-token entropy arrays, by `divisor`.

    MultiTokenEntropyEstimator writes `entropy_values` as an object column of float arrays;
    the rest are plain floats. NaNs mark rows whose entropy could not be measured and are
    preserved as NaNs so the trainer's backfill still recognises them.
    """
    if column.dtype != object:
        return column / divisor

    return column.map(lambda v: v if v is None or np.isscalar(v) else np.asarray(v, dtype=float) / divisor)


OUT_PATH.mkdir(parents=True, exist_ok=True)

manifest = {}
summaries = []

for source_file in sorted(SOURCE_PATH.glob("*.parquet")):
    dataset, model_key = source_file.stem.split("_", 1)
    assert model_key in vocab_info, f"{source_file.name}: unknown model key {model_key!r}"

    info = vocab_info[model_key]
    divisor = info["ln_vocab_size"]

    df = pd.read_parquet(source_file)
    columns = sorted(ENTROPY_COLUMNS & set(df.columns))
    assert columns, f"{source_file.name}: no entropy columns among {list(df.columns)}"

    for column in columns:
        summaries.append(
            {
                "file": source_file.name,
                "column": column,
                "raw_mean": np.nanmean(np.concatenate([np.atleast_1d(v) for v in df[column].dropna()])),
                "raw_max": np.nanmax(np.concatenate([np.atleast_1d(v) for v in df[column].dropna()])),
            }
        )
        df[column] = normalize_column(df[column], divisor)

    df.to_parquet(OUT_PATH / source_file.name, index=False)

    manifest[source_file.name] = {
        "dataset": dataset,
        "model_key": model_key,
        "normalized_columns": columns,
        "rows": len(df),
        **info,
    }
    print(f"{source_file.name:34s} rows={len(df):6d} / ln|V|={divisor:.4f}  {columns}")

(OUT_PATH / "_normalization.json").write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"\nwrote {len(manifest)} parquets + _normalization.json to {OUT_PATH}")

## Verify

Row counts must match the source exactly, every normalized value must land in `[0, 1]`,
and `raw / normalized` must be the constant `ln|V|` for that model.

In [ ]:
checks = []

for source_file in sorted(SOURCE_PATH.glob("*.parquet")):
    entry = manifest[source_file.name]
    raw = pd.read_parquet(source_file)
    normalized = pd.read_parquet(OUT_PATH / source_file.name)

    assert len(raw) == len(normalized), f"{source_file.name}: row count changed"
    assert list(raw.columns) == list(normalized.columns), f"{source_file.name}: columns changed"

    for column in entry["normalized_columns"]:
        raw_values = np.concatenate([np.atleast_1d(v) for v in raw[column].dropna()])
        new_values = np.concatenate([np.atleast_1d(v) for v in normalized[column].dropna()])

        assert raw[column].isna().sum() == normalized[column].isna().sum(), f"{source_file.name}.{column}: NaNs changed"
        assert ((new_values >= 0) & (new_values <= 1)).all(), f"{source_file.name}.{column}: outside [0, 1]"

        ratios = raw_values[new_values > 0] / new_values[new_values > 0]
        assert np.allclose(ratios, entry["ln_vocab_size"]), f"{source_file.name}.{column}: divisor mismatch"

        checks.append(
            {
                "file": source_file.name,
                "column": column,
                "raw_mean": raw_values.mean(),
                "normalized_mean": new_values.mean(),
                "normalized_max": new_values.max(),
                "ln|V|": entry["ln_vocab_size"],
            }
        )

print(f"all {len(checks)} column checks passed")
pd.DataFrame(checks)